# Solution to the Challenge for Ambitious Readers: Ollama Integration (Phi-3)
This notebook presents an example integration of a local language model run via the Ollama tool to replace the mocked `simulate_llm` function.

## 1. Installation and importing dependencies
Make sure you have the `requests` library installed and the Ollama tool running (e.g. `ollama run phi3`).

In [ ]:
import requests
import json
import time

## 2. Replacing the if/else mockup with the Ollama client
We create a new function that formulates a prompt based on the current state and sends it to the local Ollama API. We also measure the response time (latency) to highlight the difference compared to the classical planner.

In [ ]:
def simulate_llm_ollama(uav_pos, radars, status, model="phi3", temperature=0.7):
    # Formulate the prompt
    prompt = f"""
You are a UAV command system. Analyze the data below and make decisions.
Your position: {uav_pos}
Radars in the area (position, range): {radars}
Mission status: {status}

Tactical rules:
1. If UAV is closer than 1.2 * radar range, return action: "low_altitude(z_max=2)".
2. If radar detected UAV in previous step, return action: "evacuation - transmit_data()".
3. Otherwise return action: "continue - optimize_time".

Provide the answer strictly in JSON format containing 'action' and 'justification' fields.
Example correct JSON: {{"action": "your_action", "justification": "your_justification"}}
"""
    
    start_time = time.time()
    
    try:
        # Communication with Ollama API on default port 11434
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {
                    "temperature": temperature
                }
            }
        )
        latency = time.time() - start_time
        result = response.json()
        
        # Parsing the response to a dictionary object
        try:
            decision = json.loads(result["response"])
        except json.JSONDecodeError:
            decision = {"action": "continue - optimize_time", "justification": "JSON parsing error."}
            
        return decision, latency
    except Exception as e:
        latency = time.time() - start_time
        return {"action": "continue - optimize_time", "justification": f"Connection error: {e}"}, latency

## 3. Latency and Nondeterminism Experiment
We run the same scenario multiple times for the same state (in the area close to the radar). Due to temperature > 0 (nondeterminism), the model may generate differently formulated justifications and require different computation time.

In [ ]:
# Test state: UAV close to radar
uav_position = (12, 10, 5)
radars_list = [{"pos": (10, 10), "range": 3}]
mission_status = "Standard patrol flight"

print("Running the simulation 3 times to observe nondeterminism...")
for i in range(1, 4):
    print(f"\n--- Trial {i} ---")
    decision, duration = simulate_llm_ollama(uav_position, radars_list, mission_status, temperature=0.8)
    
    print(f"Received decision: {decision.get('action', decision.get('akcja', 'NONE'))}")
    print(f"Justification: {decision.get('justification', decision.get('uzasadnienie', 'NONE'))}")
    print(f"Response time (Latency): {duration:.3f} seconds")
    
print("\nConclusion: A time of a few seconds means that, e.g., for an object moving at 90 km/h (25 m/s), decision-making takes place over 25-100 meters of flight. This drastically affects safety margins.")